# Redes Neurais Recorrentes

As Redes Neurais Recorrentes são arquiteturas desenvolvidas para o processamento de dados sequenciais. Diferente das redes tradicionais, que consideram as entradas independentes, estas redes possuem conexões que transmitem informações ao longo do tempo.

Essa característica possibilita manter uma memória de informações passadas por meio de um estado oculto, que funciona como um resumo das informações anteriores.

### Formulação

A formulação matemática básica de uma célula recorrente calcula o estado oculto e a respectiva saída por meio de transformações lineares e funções de ativação não lineares.

As equações principais para cada instante de tempo são descritas abaixo:

$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh})$$

$$y_t = W_{ho} h_t + b_{ho}$$

Definição de cada variável:
* $x_t$ representa o vetor de entrada no instante de tempo $t$, com dimensão de características.
* $h_t$ representa o estado oculto no instante de tempo $t$, atuando como a memória da rede.
* $W_{ih}$ e $W_{hh}$ representam as matrizes de pesos que conectam a entrada e o estado oculto anterior ao estado atual.
* $b_{ih}$ e $b_{hh}$ representam os vetores de viés para as transformações do estado oculto.
* $y_t$ representa a saída da rede no instante de tempo $t$.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Definindo sementes para reprodutibilidade
torch.manual_seed(42)
np.random.seed(42)

# Configuração do dispositivo de processamento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo em uso:", device)

## Processamento de Sequências

Uma rede recorrente recebe uma sequência de dados e propaga a informação ao longo dos instantes de tempo. Vamos ilustrar essa dinâmica implementando uma célula básica, depois uma estrutura que realiza o loop sobre a sequência completa de tempo e finalmente comparando com a implementação nativa do PyTorch.

In [ ]:
class BasicRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.input_to_hidden = nn.Linear(input_size, hidden_size)
        self.hidden_to_hidden = nn.Linear(hidden_size, hidden_size, bias=False)
        self.activation = nn.Tanh()
        
    def forward(self, x, h_prev):
        # x: [batch_size, input_size]
        # h_prev: [batch_size, hidden_size]
        return self.activation(self.input_to_hidden(x) + self.hidden_to_hidden(h_prev))

In [ ]:
class BasicRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.cell = BasicRNNCell(input_size, hidden_size)
        
    def forward(self, x):
        # x: [batch_size, sequence_length, input_size]
        batch_size = x.size(0)
        sequence_length = x.size(1)
        
        h = torch.zeros(batch_size, self.hidden_size, device=x.device)
        outputs = []
        
        for t in range(sequence_length):
            x_t = x[:, t, :]
            h = self.cell(x_t, h)
            outputs.append(h.unsqueeze(1))
            
        return torch.cat(outputs, dim=1), h

In [ ]:
batch_size = 3
sequence_length = 8
input_size = 4
hidden_size = 5

x_dummy = torch.randn(batch_size, sequence_length, input_size)
model_custom = BasicRNN(input_size, hidden_size)
out_custom, h_custom = model_custom(x_dummy)

print("Saída do loop:", out_custom.shape)

### O Módulo `nn.RNN` do PyTorch

Em ambientes de produção e pesquisa, utilizamos a classe `torch.nn.RNN`. Essa implementação é altamente otimizada, suportando aceleração via **cuDNN** (*CUDA Deep Neural Network library*) e execução paralela sempre que possível.

Ao instanciar `nn.RNN`, os principais hiperparâmetros são:

- `input_size`: número de features esperadas em cada elemento da sequência de entrada.
- `hidden_size`: número de features no estado oculto da RNN.
- `num_layers`: número de camadas recorrentes empilhadas (*stacked RNN*).
- `batch_first`: se `True`, a entrada e a saída seguem o formato `(batch, seq_len, features)`. O padrão do PyTorch é `False`, usando `(seq_len, batch, features)`.
- `nonlinearity`: função de ativação interna da RNN, podendo ser `'tanh'` por padrão ou `'relu'`.

A chamada `forward` desse módulo espera uma entrada `x` e, opcionalmente, um estado oculto inicial `h_0`. Caso `h_0` seja omitido, o PyTorch assume automaticamente um estado inicial preenchido com zeros.

In [ ]:
rnn = nn.RNN(input_size=input_size, hidden_size=hidden_size, batch_first=True)

out, h = rnn(x_dummy)

print("Entrada:", x_dummy.shape)
print("Saída da nn.RNN:", out.shape)
print("Estado oculto final:", h.shape)

## Classificação de Séries Temporais

Para este problema, vamos construir um classificador que identifica a frequência de um sinal oscilatório ruidoso. O modelo lerá a sequência temporal completa e fará a previsão da classe correspondente no final do processamento.

In [ ]:
def generate_classification_data(num_samples=400, seq_len=50):
    half_samples = num_samples // 2
    t = np.linspace(0, 10, seq_len)
    
    # Sinais limpos
    x0 = np.sin(0.3 * t)
    x1 = np.sin(1.5 * t)
    
    data = []
    labels = []
    
    for _ in range(half_samples):
        noise = np.random.normal(0, 0.2, seq_len)
        data.append(x0 + noise)
        labels.append(0)
        
    for _ in range(half_samples):
        noise = np.random.normal(0, 0.2, seq_len)
        data.append(x1 + noise)
        labels.append(1)
        
    data = np.array(data, dtype=np.float32)
    labels = np.array(labels, dtype=np.int64)
    
    indices = np.random.permutation(num_samples)
    data = data[indices]
    labels = labels[indices]
    
    data = np.expand_dims(data, axis=-1)
    return data, labels

In [ ]:
data, labels = generate_classification_data()
print("Dimensão dos dados:", data.shape)
print("Dimensão das etiquetas:", labels.shape)

# Plotando amostras de cada classe
plt.figure(figsize=(10, 4))
idx_class_0 = np.where(labels == 0)[0][0]
idx_class_1 = np.where(labels == 1)[0][0]

plt.plot(data[idx_class_0, :, 0], label="Classe 0 - Frequência Baixa", color="blue")
plt.plot(data[idx_class_1, :, 0], label="Classe 1 - Frequência Alta", color="red")
plt.title("Amostras de Sinais Ruidosos para Classificação")
plt.xlabel("Instante de Tempo")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()

### Dataset e DataLoader

Criamos uma classe para carregar nossos dados no PyTorch e realizar o particionamento entre conjuntos de treinamento e teste.

In [ ]:
class ClassificationDataset(Dataset):
    def __init__(self, x_data, y_data):
        self.x = torch.tensor(x_data)
        self.y = torch.tensor(y_data)
        
    def __len__(self):
        return len(self.x)
        
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [ ]:
split = int(0.8 * len(data))
x_train, x_test = data[:split], data[split:]
y_train, y_test = labels[:split], labels[split:]

train_dataset = ClassificationDataset(x_train, y_train)
test_dataset = ClassificationDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Arquitetura do Classificador

A arquitetura do classificador contém uma camada recorrente nativa seguida por uma camada linear simples que projeta a representação do último passo de tempo em duas classes.

In [ ]:
class RNNClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
        
    def forward(self, x):
        # x: [batch_size, sequence_length, input_size]
        out, h_n = self.rnn(x)
        # Último passo de tempo: [batch_size, hidden_size]
        last_out = out[:, -1, :]
        logits = self.fc(last_out)
        return logits

In [ ]:
model = RNNClassifier(input_size=1, hidden_size=8, num_classes=2).to(device)
print(model)

### Otimização e Treinamento

Definimos o otimizador Adam e a função de perda de entropia cruzada para ajustar os parâmetros do modelo ao longo das épocas de treinamento.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

In [ ]:
epochs = 15
losses = []

model.train()
for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Época {epoch+1:02d}/{epochs} | Perda: {avg_loss:.4f}")

plt.figure(figsize=(8, 4))
plt.plot(losses, color="purple", marker="o")
plt.title("Evolução da Perda durante o Treinamento")
plt.xlabel("Época")
plt.ylabel("Perda")
plt.grid(True)
plt.show()

### Visualização de Resultados

Avaliamos o modelo em dados de teste inéditos para verificar a qualidade das classificações obtidas por meio de predições visuais.

In [ ]:
model.eval()
test_x, test_y = next(iter(test_loader))

with torch.no_grad():
    test_x = test_x.to(device)
    logits = model(test_x)
    predictions = torch.argmax(logits, dim=1).cpu().numpy()
    
test_x = test_x.cpu().numpy()
test_y = test_y.numpy()

plt.figure(figsize=(12, 6))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    is_correct = predictions[i] == test_y[i]
    color = "green" if is_correct else "red"
    
    plt.plot(test_x[i, :, 0], color="black", alpha=0.6)
    plt.title(f"Real: {test_y[i]} | Previsto: {predictions[i]}", color=color)
    plt.grid(True)
    
plt.tight_layout()
plt.show()

## Predição de Séries Temporais

Agora criamos um modelo recorrente para prever o próximo valor de uma série temporal contínua com base em uma janela de observações passadas.

In [ ]:
# Geração da série temporal contínua com maior frequência e ruído gaussiano
t = np.linspace(0, 50, 400)
series = np.sin(1.5 * t) + np.random.normal(0, 0.3, len(t))

plt.figure(figsize=(10, 4))
plt.plot(series, color="blue")
plt.title("Série Temporal Contínua para Predição")
plt.xlabel("Instante de Tempo")
plt.ylabel("Valor")
plt.grid(True)
plt.show()

### Preparação do Dataset de Janelas Deslizantes

Montamos as janelas temporais de tamanho fixo onde a entrada é uma sequência curta e o alvo de treinamento representa o elemento imediatamente seguinte.

In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, data_series, window_size=20):
        self.data = torch.tensor(data_series, dtype=torch.float32)
        self.window_size = window_size
        
    def __len__(self):
        return len(self.data) - self.window_size
        
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.window_size].unsqueeze(-1)
        y = self.data[idx + self.window_size].unsqueeze(-1)
        return x, y

In [ ]:
split_index = 300
train_series = series[:split_index]
test_series = series[split_index:]

window_size = 10
train_predict_dataset = TimeSeriesDataset(train_series, window_size)
test_predict_dataset = TimeSeriesDataset(test_series, window_size)

train_predict_loader = DataLoader(train_predict_dataset, batch_size=16, shuffle=True)
test_predict_loader = DataLoader(test_predict_dataset, batch_size=1, shuffle=False)

### Arquitetura do Preditor

A arquitetura do preditor utiliza a camada recorrente nativa para codificar a janela temporal em um estado latente e uma camada linear que estima o próximo valor escalar contínuo.

In [ ]:
class RNNPredictor(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
        # x: [batch_size, window_size, input_size]
        out, h_n = self.rnn(x)
        last_out = out[:, -1, :]
        prediction = self.fc(last_out)
        return prediction

In [ ]:
predictor_model = RNNPredictor(input_size=1, hidden_size=16).to(device)
print(predictor_model)

### Otimização e Treinamento do Preditor

Utilizamos o otimizador Adam e a função de perda do erro quadrático médio para o treinamento do preditor.

In [ ]:
criterion_mse = nn.MSELoss()
optimizer_predictor = optim.Adam(predictor_model.parameters(), lr=0.005)

In [ ]:
predict_epochs = 100
predict_losses = []

predictor_model.train()
for epoch in range(predict_epochs):
    epoch_loss = 0.0
    for batch_x, batch_y in train_predict_loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        
        predictions = predictor_model(batch_x)
        loss = criterion_mse(predictions, batch_y)
        
        optimizer_predictor.zero_grad()
        loss.backward()
        # Estabilização dos gradientes por meio de corte de norma
        nn.utils.clip_grad_norm_(predictor_model.parameters(), max_norm=1.0)
        optimizer_predictor.step()
        
        epoch_loss += loss.item()
        
    avg_loss = epoch_loss / len(train_predict_loader)
    predict_losses.append(avg_loss)
    print(f"Época {epoch+1:02d}/{predict_epochs} | MSE Loss: {avg_loss:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(predict_losses, color="green", marker="s")
plt.title("Evolução da Perda do Preditor")
plt.xlabel("Época")
plt.ylabel("MSE Loss")
plt.grid(True)
plt.show()

### Avaliação e Geração Autoregressiva

Avaliamos a capacidade de predição gerando os pontos futuros de forma autoregressiva. Alimentamos o modelo com a primeira janela de teste. Ele estima o primeiro ponto futuro. Em seguida, removemos o ponto mais antigo dessa janela e adicionamos a predição obtida para realizar a estimativa do passo seguinte, repetindo o ciclo recursivamente.

In [ ]:
predictor_model.eval()
autoregressive_predictions = []
true_targets = []

# Inicializamos a janela deslizante com a primeira janela do conjunto de teste
current_window = torch.tensor(test_series[:window_size], dtype=torch.float32).unsqueeze(-1).to(device)

# Geramos os pontos seguintes de forma recursiva
steps_to_predict = len(test_series) - window_size

with torch.no_grad():
    for i in range(steps_to_predict):
        # Formato de entrada esperado com dimensão de lote: [1, window_size, 1]
        x_input = current_window.unsqueeze(0)
        pred = predictor_model(x_input)
        
        pred_val = pred.item()
        autoregressive_predictions.append(pred_val)
        true_targets.append(test_series[window_size + i])
        
        # Atualizamos a janela: removemos o primeiro elemento e adicionamos a nova predição ao final
        new_point = torch.tensor([[pred_val]], dtype=torch.float32).to(device)
        current_window = torch.cat([current_window[1:], new_point], dim=0)

# Plotando os resultados comparativos de geração autoregressiva
plt.figure(figsize=(10, 4))
plt.plot(true_targets, label="Alvo Real", color="blue")
plt.plot(autoregressive_predictions, label="Geração Autoregressiva", color="orange", linestyle="--")
plt.title("Geração Autoregressiva de Pontos Futuros no Conjunto de Teste")
plt.xlabel("Passo Temporal")
plt.ylabel("Valor")
plt.legend()
plt.grid(True)
plt.show()

## Exercícios

### Exercício 1: Estado oculto

Altere o tamanho do estado oculto do preditor para 4 e para 64. Treine o modelo nas mesmas condições e compare o resultado visual das predições para cada caso.

### Exercício 2: Ruído

Aumente a quantidade de ruído na geração do conjunto de dados de classificação. Execute o treinamento novamente e analise como a taxa de acerto do classificador reage ao aumento de ruído nos sinais.